In [20]:
import re

def extract_par_utterances_with_duration(cha_file_path):
    with open(cha_file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    data = []

    for line in lines:
        if line.startswith("*PAR:"):
            # Extract timestamp
            timestamp_match = re.search(r'\x15(\d+)_(\d+)\x15', line)
            if not timestamp_match:
                continue

            start, end = map(int, timestamp_match.groups())
            duration = (end - start) / 1000.0  # convert ms to seconds

            # Remove timestamp and *PAR: tag
            text = re.sub(r'\x15\d+_\d+\x15', '', line)
            text = text.replace("*PAR:", "").strip()

            # Remove CHAT annotations (non-speech)
            text = re.sub(r'\[\+[^\]]*\]', '', text)     # [+ exc], [+ laugh], etc.
            text = re.sub(r'\[//\]|\[/\]', '', text)     # retracings
            text = re.sub(r'\[\*\s?[^\]]*\]', '', text)  # [* m:], etc.

            # Remove [: over] and similar
            text = re.sub(r'\[:[^\]]*\]', '', text)

            # Remove &=laughs and similar
            text = re.sub(r'&=\S+', '', text)

            # Remove @tags like chicken@q → chicken
            text = re.sub(r'@[\w]+', '', text)
            
            # Remove +< but keep inserted speech
            text = text.replace('+<', '')
            
            # Remove symbols like ‡
            text = text.replace('‡', '')

            text = re.sub(r'\&[-+]\S+', '', text)        # &-uh, &+lit
            text = re.sub(r'\+\.+', '', text)            # +...
            text = re.sub(r'\+\/', '', text)             # +/
            text = re.sub(r'\b(xxx|www)\b', '', text, flags=re.IGNORECASE)

            # Keep contents inside < > but remove brackets
            text = re.sub(r'<([^>]*)>', r'\1', text)

            # Remove underscores
            text = text.replace('_', '')
            text = text.replace('+', '')
            text = text.replace('/', '')

            # Remove short parentheses like (g), (be), etc.
            def paren_filter(match):
                content = match.group(1)
                if len(content.strip()) <= 5:
                    return ''
                else:
                    return f'({content})'
            text = re.sub(r'\(([^)]*)\)', paren_filter, text)

            # Normalize whitespace
            text = re.sub(r'\s+', ' ', text).strip()

            # Remove trailing " ."
            text = re.sub(r'\s+\.$', '', text)

            data.append({
                "utterance": text,
                "duration": duration,
            })

    return data

In [21]:
import os

subfolders = ['cookie',]

dementia_dir = "./Pitt/Dementia"
control_dir = "./Pitt/Control/"

data = []
labels = []

for folder in subfolders:
    folder_dir = os.path.join(dementia_dir, folder)
    for file in os.listdir(folder_dir):
        if '.cha' not in file:
            continue
        speech_data = extract_par_utterances_with_duration(os.path.join(folder_dir, file))
        data.append(speech_data)
        labels.append(1)

for folder in subfolders:
    folder_dir = os.path.join(control_dir, folder)
    for file in os.listdir(folder_dir):
        if '.cha' not in file:
            continue
        speech_data = extract_par_utterances_with_duration(os.path.join(folder_dir, file))
        data.append(speech_data)
        labels.append(0)


In [22]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import gensim.downloader as api
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

class AlzheimerDataset(Dataset):
    """Custom dataset for Alzheimer's detection"""
    
    def __init__(self, trials_data, labels, word2vec_model, max_utterances=20, word2vec_dim=300):
        self.trials_data = trials_data
        self.labels = labels
        self.word2vec_model = word2vec_model
        self.max_utterances = max_utterances
        self.word2vec_dim = word2vec_dim
        
    def __len__(self):
        return len(self.trials_data)
    
    def preprocess_text(self, text: str) -> List[str]:
        """Clean and tokenize text"""
        text = re.sub(r'[^a-zA-Z\s]', '', text.lower())
        words = text.split()
        return words
    
    def get_utterance_embedding(self, utterance: str) -> np.ndarray:
        """Get word2vec embedding for an utterance with mean, max, and std"""
        words = self.preprocess_text(utterance)
        embeddings = []
        
        for word in words:
            if word in self.word2vec_model:
                embeddings.append(self.word2vec_model[word])
        
        if embeddings:
            emb_array = np.array(embeddings)
            mean_emb = np.mean(emb_array, axis=0)
            max_emb = np.max(emb_array, axis=0)
            std_emb = np.std(emb_array, axis=0)
            return np.concatenate([mean_emb, max_emb, std_emb])  # shape (900,)
        else:
            return np.zeros(self.word2vec_dim * 3)  # shape (900,)
    
    def __getitem__(self, idx):
        trial = self.trials_data[idx]
        label = self.labels[idx]
        
        features = []
        for utterance_data in trial:
            utterance = utterance_data['utterance']
            duration = utterance_data['duration']
            
            # Get word2vec embedding
            embedding = self.get_utterance_embedding(utterance)
            
            # Concatenate embedding with duration
            feature_vector = np.concatenate([embedding, [duration]])
            features.append(feature_vector)
        
        # Handle empty trials
        if len(features) == 0:
            features = [np.zeros(self.word2vec_dim + 1)]
        
        # Convert to tensor
        features = torch.FloatTensor(features)
        
        # Store original length before padding/truncating
        original_length = len(features)
        
        # Pad or truncate
        if len(features) > self.max_utterances:
            features = features[:self.max_utterances]
            original_length = self.max_utterances
        elif len(features) < self.max_utterances:
            padding = torch.zeros(self.max_utterances - len(features), self.word2vec_dim*3 + 1)
            features = torch.cat([features, padding], dim=0)
        
        return features, torch.FloatTensor([label]), original_length


class AlzheimerLSTM(nn.Module):
    """LSTM model for Alzheimer's detection"""
    
    def __init__(self, input_dim=901, hidden_dim=256, num_layers=3, dropout=0.3):
        super(AlzheimerLSTM, self).__init__()
        
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_dim, 
            hidden_dim, 
            num_layers, 
            batch_first=True, 
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dense layers
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim*2, 64)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x, lengths):
        batch_size = x.size(0)
        
        # Ensure minimum length of 1 for all sequences
        lengths = torch.clamp(lengths, min=1)
        
        # Pack padded sequence
        packed_input = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        # LSTM forward pass
        packed_output, (hidden, cell) = self.lstm(packed_input)
        
        # Use the last hidden state from both directions
        # hidden shape: (num_layers * 2, batch_size, hidden_dim)
        hidden = hidden[-2:].transpose(0, 1).contiguous().view(batch_size, -1)
        
        # Dense layers
        out = self.dropout(hidden)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout2(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        
        return out.squeeze()


class AlzheimerDetectionPipeline:
    def __init__(self, max_utterances=20, word2vec_dim=300, device=None):
        self.max_utterances = max_utterances
        self.word2vec_dim = word2vec_dim
        self.word2vec_model = None
        self.device = device if device else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
    def load_word2vec(self):
        """Load pre-trained Word2Vec model"""
        print("Loading Word2Vec model...")
        self.word2vec_model = api.load('word2vec-google-news-300')
        print("Word2Vec model loaded successfully!")
        
    def train_epoch(self, model, dataloader, criterion, optimizer):
        """Train for one epoch"""
        model.train()
        total_loss = 0
        
        for batch_features, batch_labels, batch_lengths in dataloader:
            batch_features = batch_features.to(self.device)
            batch_labels = batch_labels.to(self.device).squeeze()
            batch_lengths = torch.tensor(batch_lengths, dtype=torch.long)
            
            optimizer.zero_grad()
            
            outputs = model(batch_features, batch_lengths)
            loss = criterion(outputs, batch_labels)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        return total_loss / len(dataloader)
    
    def validate_epoch(self, model, dataloader, criterion):
        """Validate for one epoch"""
        model.eval()
        total_loss = 0
        all_predictions = []
        all_labels = []
        
        with torch.no_grad():
            for batch_features, batch_labels, batch_lengths in dataloader:
                batch_features = batch_features.to(self.device)
                batch_labels = batch_labels.to(self.device).squeeze()
                batch_lengths = torch.tensor(batch_lengths, dtype=torch.long)
                
                outputs = model(batch_features, batch_lengths)
                loss = criterion(outputs, batch_labels)
                
                total_loss += loss.item()
                
                all_predictions.extend(outputs.cpu().numpy())
                all_labels.extend(batch_labels.cpu().numpy())
        
        return total_loss / len(dataloader), np.array(all_predictions), np.array(all_labels)
    
    def train_fold(self, train_dataset, val_dataset, epochs=100, batch_size=16, 
                   learning_rate=0.001, patience=10):
        """Train model for one fold"""
        
        # Create data loaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Initialize model
        model = AlzheimerLSTM(input_dim=self.word2vec_dim*3 + 1).to(self.device)
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
        # Early stopping variables
        best_val_loss = float('inf')
        patience_counter = 0
        best_model_state = None
        
        train_losses = []
        val_losses = []
        
        for epoch in range(epochs):
            # Train
            train_loss = self.train_epoch(model, train_loader, criterion, optimizer)
            
            # Validate
            val_loss, val_predictions, val_labels = self.validate_epoch(model, val_loader, criterion)
            
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            
            # Early stopping
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict().copy()
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        # Load best model
        model.load_state_dict(best_model_state)
        
        return model, {'train_losses': train_losses, 'val_losses': val_losses}
    
    def evaluate_model(self, model, dataset):
        """Evaluate model performance"""
        dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
        
        model.eval()
        all_predictions = []
        all_probabilities = []
        all_labels = []
        
        with torch.no_grad():
            for batch_features, batch_labels, batch_lengths in dataloader:
                batch_features = batch_features.to(self.device)
                batch_lengths = torch.tensor(batch_lengths, dtype=torch.long)
                
                outputs = model(batch_features, batch_lengths)
                
                all_probabilities.extend(outputs.cpu().numpy())
                all_predictions.extend((outputs > 0.5).float().cpu().numpy())
                all_labels.extend(batch_labels.squeeze().numpy())
        
        # Calculate metrics
        all_predictions = np.array(all_predictions)
        all_probabilities = np.array(all_probabilities)
        all_labels = np.array(all_labels)
        
        metrics = {
            'accuracy': accuracy_score(all_labels, all_predictions),
            'precision': precision_score(all_labels, all_predictions),
            'recall': recall_score(all_labels, all_predictions),
            'f1': f1_score(all_labels, all_predictions),
            'auc': roc_auc_score(all_labels, all_probabilities)
        }
        
        return metrics
    
    def cross_validate(self, trials_data: List[List[Dict]], labels: List[int], 
                      n_splits: int = 5, epochs: int = 100, batch_size: int = 16) -> Dict:
        """Perform cross-validation"""
        print("Starting cross-validation...")
        
        # Load Word2Vec model
        if self.word2vec_model is None:
            self.load_word2vec()
        
        # Initialize cross-validation
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        
        # Store results
        fold_results = []
        all_metrics = {
            'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []
        }
        
        # Perform cross-validation
        for fold, (train_idx, val_idx) in enumerate(skf.split(trials_data, labels)):
            print(f"Training fold {fold + 1}/{n_splits}...")
            
            # Split data
            train_trials = [trials_data[i] for i in train_idx]
            val_trials = [trials_data[i] for i in val_idx]
            train_labels = [labels[i] for i in train_idx]
            val_labels = [labels[i] for i in val_idx]
            
            # Create datasets
            train_dataset = AlzheimerDataset(
                train_trials, train_labels, self.word2vec_model, 
                self.max_utterances, self.word2vec_dim
            )
            val_dataset = AlzheimerDataset(
                val_trials, val_labels, self.word2vec_model, 
                self.max_utterances, self.word2vec_dim
            )
            
            # Train model
            model, history = self.train_fold(train_dataset, val_dataset, epochs, batch_size)
            
            # Evaluate model
            metrics = self.evaluate_model(model, val_dataset)
            fold_results.append(metrics)
            
            # Store metrics
            for metric_name, value in metrics.items():
                all_metrics[metric_name].append(value)
            
            print(f"Fold {fold + 1} - Accuracy: {metrics['accuracy']:.4f}, "
                  f"F1: {metrics['f1']:.4f}, AUC: {metrics['auc']:.4f}")
        
        # Calculate mean and std for each metric
        summary_metrics = {}
        for metric_name, values in all_metrics.items():
            summary_metrics[f'{metric_name}_mean'] = np.mean(values)
            summary_metrics[f'{metric_name}_std'] = np.std(values)
        
        return {
            'fold_results': fold_results,
            'summary_metrics': summary_metrics,
            'all_metrics': all_metrics
        }
    
    def print_results(self, cv_results: Dict):
        """Print cross-validation results"""
        print("\n" + "="*60)
        print("CROSS-VALIDATION RESULTS")
        print("="*60)
        
        summary = cv_results['summary_metrics']
        
        print(f"Accuracy:  {summary['accuracy_mean']:.4f} ± {summary['accuracy_std']:.4f}")
        print(f"Precision: {summary['precision_mean']:.4f} ± {summary['precision_std']:.4f}")
        print(f"Recall:    {summary['recall_mean']:.4f} ± {summary['recall_std']:.4f}")
        print(f"F1-Score:  {summary['f1_mean']:.4f} ± {summary['f1_std']:.4f}")
        print(f"AUC-ROC:   {summary['auc_mean']:.4f} ± {summary['auc_std']:.4f}")

In [19]:
pipeline = AlzheimerDetectionPipeline(max_utterances=20)
results = pipeline.cross_validate(data, labels, n_splits=5)
pipeline.print_results(results)


Using device: cpu
Starting cross-validation...
Loading Word2Vec model...
[--------------------------------------------------] 1.4% 23.4/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==------------------------------------------------] 5.9% 98.1/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[====----------------------------------------------] 9.8% 163.2/1662.8MB downloaded

...

[===============-----------------------------------] 31.7% 527.6/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[========================--------------------------] 48.3% 803.7/1662.8MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 1662.8/1662.8MB downloaded
Word2Vec model loaded successfully!
Training fold 1/5...
Early stopping at epoch 20
Fold 1 - Accuracy: 0.7117, F1: 0.7377, AUC: 0.8032
Training fold 2/5...
Early stopping at epoch 25
Fold 2 - Accuracy: 0.7568, F1: 0.7568, AUC: 0.8144
Training fold 3/5...
Early stopping at epoch 24
Fold 3 - Accuracy: 0.7364, F1: 0.7786, AUC: 0.8125
Training fold 4/5...
Early stopping at epoch 24
Fold 4 - Accuracy: 0.7091, F1: 0.7037, AUC: 0.8182
Training fold 5/5...
Early stopping at epoch 30
Fold 5 - Accuracy: 0.7364, F1: 0.7752, AUC: 0.8220

CROSS-VALIDATION RESULTS
Accuracy:  0.7301 ± 0.0177
Precision: 0.7815 ± 0.0503
Recall:    0.7317 ± 0.0814
F1-Score:  0.7504 ± 0.0275
AUC-ROC:   0.8140 ± 0.0064
